In [ ]:
# Practice Problems Day 2

### Author: Joanna Bieri (solutions)

## Introduction to Machine Learning

#### University of Redlands - DATA 301
#### Prof: Joanna Bieri [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
#### Day 2 Practice - SOLUTIONS
#### [Class Website](https://joannabieri.com/machine_learning.html)

---

**These are my solutions.** Try every problem yourself first, then come here to check your work. There is more than one right way to write the code, and your words will be different from mine, that is fine! If your numbers do not match, look for the difference before you look at my code.

**Reading:** Geron, chapter 4, the sections on **Polynomial Regression** and **Learning Curves**

GOALS:

1. Get comfortable telling underfitting and overfitting apart.
2. Practice reading a learning curve.
3. Start noticing that the number a model reports is not the same as how good it is.

---

## Problem 1: Make some data of your own

Build a dataset the same way we did in class, but change the underlying function.
Use something that is **not** a parabola. A cubic, a sine wave, whatever you like.

Add noise, plot it, and say in one sentence what the true relationship is.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=0)

m = 200
X = 6 * rng.random((m, 1)) - 3
y = 0.3 * X**3 - X + 2 + rng.standard_normal((m, 1))

plt.figure(figsize=(6, 4))
plt.plot(X, y, "b.")
plt.xlabel("$x$"); plt.ylabel("$y$", rotation=0); plt.grid()
plt.show()

I used a cubic. The true relationship is $y = 0.3x^3 - x + 2$, plus noise with a standard deviation of 1.

BEWARE - make sure you actually add the noise! Without it every model looks perfect and Problem 3 does not show you anything.

---

## Problem 2: Find the degree that fits it

Fit polynomial models of several different degrees to your data from Problem 1 and
plot them on top of the points.

**2a.** Which degree looks best to your eye?

**2b.** Fit a degree that is clearly too low and one that is clearly too high. Show
all three on one plot.

**2c.** In words, what is going wrong in each of the two bad ones? Be specific about
what you see in the plot, not just "it is underfitting".

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression

X_new = np.linspace(-3, 3, 100).reshape(100, 1)

plt.figure(figsize=(6, 4))
plt.plot(X, y, "b.")

for degree, style in [(1, "r-"), (3, "g-"), (40, "m-")]:

    poly_features = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly = poly_features.fit_transform(X)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_poly)

    lin_reg = LinearRegression()
    lin_reg.fit(X_scaled, y)

    X_new_poly = poly_features.transform(X_new)
    X_new_scaled = scaler.transform(X_new_poly)
    y_pred = lin_reg.predict(X_new_scaled)

    plt.plot(X_new, y_pred, style, linewidth=2, label=f"degree {degree}")

plt.axis([-3, 3, y.min()-1, y.max()+1]); plt.legend(); plt.grid()
plt.show()

**2a.** Degree 3, which is what I used to build the data.

**2b.** Degree 1 is too low and degree 40 is too high. All three are on the plot above.

**2c.** The important thing here is to describe what you **see in the plot**, not just the name for it.

- **Degree 1** cannot bend enough. The line is above the points at the ends and below them in the middle, so the errors come in runs instead of being scattered around the line. That pattern tells you the model is too simple.
- **Degree 40** goes through the data but wiggles hard between the points, especially at the edges where there is less data to pin it down. Ask it to predict at $x = 2.9$ and you get something wild. It is fitting the noise.

---

## Problem 3: Learning curves

Use the `plot_learning_curve` function from the Day 2 notes on two of your models:
the one you think is right, and the one that is clearly too flexible.

**3a.** Paste both plots.

**3b.** For each one, say what you see: are the curves high or low, together or
apart, still moving or flattened out?

**3c.** For the too-flexible model, would collecting more data help? How can you
tell from the plot rather than from knowing the answer already?

In [ ]:
from sklearn.model_selection import learning_curve
from sklearn.pipeline import make_pipeline

def plot_learning_curve(model, X, y, title):
    train_sizes, train_scores, valid_scores = learning_curve(
        model, X, y.ravel(), train_sizes=np.linspace(0.01, 1.0, 40), cv=5,
        scoring="neg_root_mean_squared_error")
    train_errors = -train_scores.mean(axis=1)
    valid_errors = -valid_scores.mean(axis=1)

    plt.figure(figsize=(6, 4))
    plt.plot(train_sizes, train_errors, "r-+", linewidth=2, label="training set")
    plt.plot(train_sizes, valid_errors, "b-", linewidth=3, label="validation set")
    plt.xlabel("training set size"); plt.ylabel("RMSE")
    plt.title(title); plt.legend(); plt.grid(); plt.axis([0, 200, 0, 5])
    plt.show()

good_model = make_pipeline(
    PolynomialFeatures(degree=3, include_bias=False),
    LinearRegression())

plot_learning_curve(good_model, X, y, "Degree 3, about right")

In [ ]:
too_flexible = make_pipeline(
    PolynomialFeatures(degree=40, include_bias=False),
    StandardScaler(),
    LinearRegression())

plot_learning_curve(too_flexible, X, y, "Degree 40, too flexible")

**3b.**

- **Degree 3:** both curves settle down close to 1.0, which is the noise we added, and they are close together. That is about as good as anyone can do on this data. You cannot beat the noise.
- **Degree 40:** the training curve ends a bit below the noise (about 0.96) and the validation curve sits well above it (about 1.2). That gap is the overfitting.

**3c.** Yes, more data would help the degree 40 model. You can tell from the plot because **the validation curve is still trending down at the right side**. It has not flattened out, so the model still has something to gain from more data.

Compare that to underfitting, where both curves have gone flat and more data changes nothing.

---

## Problem 4: The number is not the model

Take your most overfit model from Problem 2 and compute its error **on the training
data only**.

**4a.** What error do you get? Is it good?

**4b.** A classmate shows you this number and says their model is excellent. What is
the one question you would ask them?

**4c.** Connect this to Day 1. Which rule from our first class is this problem an
example of?


---

In [ ]:
from sklearn.metrics import root_mean_squared_error

poly_features = PolynomialFeatures(degree=40, include_bias=False)
X_poly = poly_features.fit_transform(X)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_poly)

lin_reg = LinearRegression()
lin_reg.fit(X_scaled, y)

y_predicted = lin_reg.predict(X_scaled)
rmse_train = root_mean_squared_error(y, y_predicted)

print("degree 40, RMSE on the TRAINING data:", round(rmse_train, 3))
print("the noise floor is about 1.0, so anything below that is memorization")

**4a.** I get **0.979**, just under the noise level of 1.0. Your number will be different, it depends on the function and the degree you picked.

The number is not spectacular. What matters is that it is **at or below the noise level at all**. You cannot honestly do better than the noise, because the noise is random and there is nothing in it to learn. A model that scores at or under that level is fitting the random wobbles in these 200 points, and those wobbles will be different in new data.

NOTE - with 200 points the effect is small. If you picked a higher degree or used fewer points you probably saw a much smaller training error. Either way, a low training error is not evidence that the model is good.

**4b.** *"What data did you measure that on?"* Any version of this works: did you hold data out, what was the validation error, did you test on data the model had not seen?

**4c.** This is the Day 1 rule: **you only get to use your test set once**. More broadly, a score on the data the model was trained on is not evidence of how it will do on new data.

---

## Problem 5: A judgment call

You are handed a model at work. Training error is 0.4 and validation error is 0.45.
Both curves in the learning curve have flattened out.

**5a.** Is this model underfitting, overfitting, or neither? How do you know?

**5b.** Your boss asks you to make it better. Based only on what you can see here,
what would you try first, and what would you *not* bother trying? Explain why.

*There is more than one reasonable answer. I care about your reasoning.*

**5a.** Not overfitting. The training and validation errors are almost the same (0.4 and 0.45), so there is no real gap. Is it underfitting? The plot alone cannot tell you! It depends on what a good error is for this problem. If the noise in the data is about 0.4, this model is doing as well as anything can. If the noise is 0.05, it is underfitting badly. The best answer says this and asks what error is actually possible.

**5b.** What **not** to bother with: **collecting more data.** Both curves are flat, so more of the same data will not change anything. This is the reflex answer, and here it is the wrong one.

What to try: a more flexible model, better features, or find out what error is possible on this problem before spending more effort.